# Advanced RAG Pipeline: Document Embedding and Retrieval

This notebook demonstrates the core phase of building an **Advanced Retrieval-Augmented Generation (RAG)** pipeline: **Document Embedding Generation**.

To maintain a clean, modular, and production-ready codebase, the document loading and text chunking logic have been abstracted into standalone Python modules (`config.py`, `loader.py`, and `splitter.py`).

Here, we will:
1. **Load** the PDF document (`nodejs.pdf`) using our custom `loader` module.
2. **Chunk** the extracted pages into semantic text segments using our `splitter` module.
3. **Embed** the text chunks using a state-of-the-art, local `sentence-transformers` model.
4. **Search**: Implement a simple semantic search to retrieve the most relevant text chunks for a sample user query using vector similarity.

In [ ]:
# Import the modular components and external libraries
from pathlib import Path
import numpy as np
from sentence_transformers import SentenceTransformer

# Import our custom modular pipeline parts
import config
from loader import load_document
from splitter import split_records

print("Modules imported successfully!")
print(f"Target document directory: {config.DOCS_DIR.resolve()}")
print(f"Configured embedding model: {config.EMBEDDING_MODEL_NAME}")

## Step 1: Load and Parse the Document

We use our custom `loader.py` module to parse the PDF document page-by-page. This extracts the text while preserving metadata about the source document and page number.

In [ ]:
# Load the documents from the configured directory
pdf_path = config.DOCS_DIR / "nodejs.pdf"

if not pdf_path.exists():
    print(f"Error: Could not find {pdf_path}. Please make sure it is in the document folder.")
else:
    # Load raw page-by-page records
    raw_records = load_document(pdf_path)
    print(f"Loaded {len(raw_records)} pages from {pdf_path.name}")
    
    # Peek at the first page
    if raw_records:
        print("\n--- Sample Page 1 Metadata ---")
        print(f"Source: {raw_records[0]['source']}")
        print(f"Page: {raw_records[0]['page']}")
        print(f"Character count: {len(raw_records[0]['text'])}")
        print("\n--- Sample Page 1 Text (First 300 chars) ---")
        print(raw_records[0]['text'][:300] + "...")

## Step 2: Semantic Text Chunking

To ensure the text fits within the embedding model's context window and to improve retrieval relevance, we split the pages into smaller, overlapping chunks using the `splitter.py` module (which leverages LangChain's `RecursiveCharacterTextSplitter`).

In [ ]:
# Chunk the page records into smaller text passages
# We use a chunk size of 600 characters with a 100-character overlap
chunks = split_records(raw_records, chunk_size=600, chunk_overlap=100)

print(f"Successfully generated {len(chunks)} text chunks.")

# Let's inspect a sample chunk to verify metadata preservation
if chunks:
    sample_idx = min(12, len(chunks) - 1)  # Inspect chunk 12 or the last chunk
    print(f"\n--- Sample Chunk {sample_idx} ---")
    print(f"Source Document: {chunks[sample_idx]['source']}")
    print(f"Original Page  : {chunks[sample_idx]['page']}")
    print(f"Chunk ID       : {chunks[sample_idx]['chunk_id']}")
    print(f"Text Content   :\n{chunks[sample_idx]['text']}")

## Step 3: Generate Text Embeddings using Sentence Transformers

Now, we initialize our local `sentence-transformers` embedding model. We will use the configured `all-MiniLM-L6-v2` model, which maps sentences and paragraphs to a 384-dimensional dense vector space.

These dense vectors capture the semantic meaning of the text, enabling us to perform semantic search beyond simple keyword matching.

In [ ]:
print(f"Loading embedding model: {config.EMBEDDING_MODEL_NAME}...")
# Initialize the sentence transformer model
# This will download the model weights (only on the first run) and load it onto CPU/GPU
model = SentenceTransformer(config.EMBEDDING_MODEL_NAME)
print("Model loaded successfully!")

# Extract just the text from our chunks to pass to the model
texts = [chunk["text"] for chunk in chunks]

print(f"Generating embeddings for {len(texts)} chunks...")
# Generate the embeddings
embeddings = model.encode(texts, show_progress_bar=True, convert_to_numpy=True)

print("\nEmbeddings generated successfully!")
print(f"Embeddings matrix shape: {embeddings.shape}")
print(f"Dimension of each embedding vector: {embeddings.shape[1]}")

# Add the generated embeddings back to our chunk records for retrieval usage
for idx, chunk in enumerate(chunks):
    chunk["embedding"] = embeddings[idx]

## Step 4: Semantic Search Demonstration

To prove the power of our embeddings, let's implement a simple semantic search function. We will calculate the **cosine similarity** between a user query's embedding and all the document chunk embeddings, returning the top matches.

In [ ]:
# def semantic_search(query: str, top_k: int = 3) -> list[dict]:
#     """
#     Computes cosine similarity between a query and all document chunks
#     to retrieve the top K most semantically relevant chunks.
#     """
#     print(f"\nUser Query: '{query}'")
    
#     # 1. Generate the embedding for the query
#     query_embedding = model.encode(query, convert_to_numpy=True)
    
#     # 2. Compute cosine similarities
#     # Cosine similarity = (A . B) / (||A|| * ||B||)
#     # Since sentence-transformer embeddings are typically normalized, we can use dot product
#     # but to be safe and clear, we calculate the standard cosine similarity
#     similarities = []
#     for chunk in chunks:
#         chunk_emb = chunk["embedding"]
        
#         # Dot product
#         dot_product = np.dot(query_embedding, chunk_emb)
#         # Norms
#         norm_query = np.linalg.norm(query_embedding)
#         norm_chunk = np.linalg.norm(chunk_emb)
        
#         # Cosine similarity score
#         score = dot_product / (norm_query * norm_chunk)
#         similarities.append((score, chunk))
        
#     # 3. Sort by similarity score in descending order
#     similarities.sort(key=lambda x: x[0], reverse=True)
    
#     # 4. Return top_k results
#     return similarities[:top_k]

# # Run a sample query search!
# sample_query = "What is Express and how do we set up a server?"
# search_results = semantic_search(sample_query, top_k=3)

# print("\n=== TOP SEMANTIC SEARCH RESULTS ===")
# for rank, (score, chunk) in enumerate(search_results, 1):
#     print(f"\n[Rank {rank}] Similarity Score: {score:.4f}")
#     print(f"Source: {chunk['source']} | Page: {chunk['page']} | Chunk ID: {chunk['chunk_id']}")
#     print("-" * 50)
#     print(chunk["text"])
#     print("-" * 50)

In [ ]:
from dotenv import load_dotenv

load_dotenv()

QDRANT_ENDPOINT=os.getenv("QDRANT_ENDPOINT") 
QDRANT_API_KEY=os.getenv("QDRANT_API_KEY")

# Fail loud if secrets missing — better than a confusing crash later
assert QDRANT_ENDPOINT, "QDRANT_ENDPOINT not found in .env"
assert QDRANT_API_KEY,  "QDRANT_API_KEY not found in .env"

In [ ]:
import os

from qdrant_client import QdrantClient
from qdrant_client.models import Distance,VectorParams,PointStruct

In [ ]:
client = QdrantClient(
    url=QDRANT_ENDPOINT,
    api_key=QDRANT_API_KEY,
)
print("Connected to Qdrant Cloud")

In [ ]:
vector_size = embeddings.shape[1]

client.recreate_collection(
    collection_name=config.COLLECTION_NAME,
    vectors_config=VectorParams(size=vector_size,distance=Distance.COSINE)
)

print(f"Collection '{config.COLLECTION_NAME}' ready (size={vector_size}, COSINE)")

In [ ]:
points = []

for idx,chunk in enumerate(chunks):
    points.append(
        PointStruct(
            id=idx,
            vector=chunk["embedding"].tolist(),
            payload={
                "text": chunk["text"],
                "source":chunk["source"],
                "page":chunk["page"],
                "chunk_id":chunk["chunk_id"]
            }
        )
    )

client.upsert(
    collection_name=config.COLLECTION_NAME,
    points=points,
    wait=True
)

count = client.count(collection_name=config.COLLECTION_NAME).count

print(f"Uploaded. Points in collection: {count}")

In [ ]:
# print(len(raw_records), len(chunks), embeddings.shape)

In [ ]:
def qdrant_search(query:str,top_k:int=3):
        # Embed query with the SAME model used for chunks (must share vector space)
    query_vector = model.encode(query,convert_to_numpy=True).tolist()
    results = client.query_points(
        collection_name=config.COLLECTION_NAME,
        query=query_vector,
            limit=top_k,
            with_payload=True
        ).points

    return results


In [ ]:
# Same query as the old numpy search — compare results
sample_query = "What is Express and how do we set up a server?"
hits = qdrant_search(sample_query, top_k=3)

print(f"Query: {sample_query}\n")
for rank, hit in enumerate(hits, 1):
    print(f"[Rank {rank}] score={hit.score:.4f} | page {hit.payload['page']} | chunk {hit.payload['chunk_id']}")
    print("-" * 50)
    print(hit.payload["text"][:300])
    print("-" * 50)

In [ ]:
from rank_bm25 import BM25Okapi

# Tokenize every chunk (lowercase + whitespace split — simple, good enough to start)
tokenized_corpus = [chunk["text"].lower().split() for chunk in chunks]

# Build the BM25 index once
bm25 = BM25Okapi(tokenized_corpus)
print(f"BM25 index built over {len(tokenized_corpus)} chunks")


In [ ]:
def bm25_search(query: str, top_k: int = 3):
    tokens = query.lower().split()           # tokenize query the SAME way
    scores = bm25.get_scores(tokens)         # one score per chunk (numpy array, len 406)
    top_idx = np.argsort(scores)[::-1][:top_k]  # indices of highest scores
    return [(int(i), float(scores[i]), chunks[i]) for i in top_idx]


# Try a keyword-heavy query where exact tokens matter
sample_query = "npm install express@4.16.4"
for rank, (i, score, chunk) in enumerate(bm25_search(sample_query), 1):
    print(f"[Rank {rank}] bm25={score:.3f} | page {chunk['page']} | chunk {chunk['chunk_id']}")
    print(chunk["text"][:200])
    print("-" * 50)



In [ ]:
def hybrid_search(query: str, top_k: int = 5, k_rrf: int = 60, pool: int = 20):
    # 1. Get a pool of candidates from BOTH retrievers
    dense_hits = qdrant_search(query, top_k=pool)          # list of Qdrant points
    bm25_hits  = bm25_search(query, top_k=pool)            # list of (idx, score, chunk)

    # 2. Build {chunk_id -> rrf_score}, accumulating across both lists
    scores = {}

    # dense: rank from enumerate order (already sorted best-first)
    for rank, hit in enumerate(dense_hits):
        cid = hit.payload["chunk_id"]
        scores[cid] = scores.get(cid, 0) + 1 / (k_rrf + rank)

    # bm25: same idea
    for rank, (idx, _bm25score, chunk) in enumerate(bm25_hits):
        cid = chunk["chunk_id"]
        scores[cid] = scores.get(cid, 0) + 1 / (k_rrf + rank)

    # 3. Sort by fused score, take top_k
    ranked_ids = sorted(scores, key=scores.get, reverse=True)[:top_k]

    # 4. Look up the chunk text by chunk_id for display
    by_cid = {c["chunk_id"]: c for c in chunks}
    return [(cid, scores[cid], by_cid[cid]) for cid in ranked_ids]


sample_query = "How do I set up an Express server with npm?"
print(f"Query: {sample_query}\n")
for rank, (cid, score, chunk) in enumerate(hybrid_search(sample_query), 1):
    print(f"[Rank {rank}] rrf={score:.4f} | page {chunk['page']} | chunk {cid}")
    print(chunk["text"][:200])
    print("-" * 50)

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

api_key=os.getenv("GROQ_API_KEY")

llm_client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1",
)

In [ ]:
def rag_answer(query:str,top_k:int=5):
    hits = hybrid_search(query,top_k=top_k)

    context = "\n\n".join(
        f"[Source{i+1} | page {chunk['page']}]\n{chunk['text']}"
        for i, (cid,score,chunk) in enumerate(hits)
    )


    system_prompt = (
        "You are a helpful assistant answering questions about knowledge. "
        "Answer ONLY using the provided context. "
        "If the answer is not in the context, say 'I don't know based on the document.' "
        "Cite the page number(s) you used like [page 37]."
    )

    user_prompt = f"Context:\n{context}\n\nQuestion: {query}"

    response = llm_client.chat.completions.create(
         model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.1,   # low = factual, less creative drift
    )

    return response.choices[0].message.content, hits

In [ ]:
# Try it — the full pipeline end to end
query = "Who is sourov?"
answer, sources = rag_answer(query)

print("=== QUERY ===")
print(query)

print("=== ANSWER ===")
print(answer)
print("\n=== SOURCES USED ===")
for i, (cid, score, chunk) in enumerate(sources, 1):
    print(f"[{i}] page {chunk['page']} | chunk {cid}")

In [ ]:
from sentence_transformers import CrossEncoder

In [ ]:
# Cross-encoder: scores (query, doc) PAIRS — slow but accurate. Run on small pools only.
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Reranker loaded")


In [49]:
def rerank(query: str, candidates: list, top_k: int = 5):
    # candidates = list of (cid, rrf_score, chunk) from hybrid_search
    pairs = [(query, c[2]["text"]) for c in candidates]   # (query, chunk_text) pairs
    scores = reranker.predict(pairs)                       # true relevance score each
    ranked = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
    return [(c[0], float(s), c[2]) for s, c in ranked[:top_k]]


In [50]:
# Prove it fixes the ordering: pull WIDE from hybrid, then rerank down
query = "How do I set up an Express server with npm?"
pool = hybrid_search(query, top_k=20)     # wide, cheap
top = rerank(query, pool, top_k=5)        # narrow, accurate

print(f"Query: {query}\n")
for rank, (cid, score, chunk) in enumerate(top, 1):
    print(f"[Rank {rank}] rerank={score:.3f} | page {chunk['page']} | chunk {cid}")
    print(chunk["text"][:160])
    print("-" * 50)


Batches: 100%|██████████| 1/1 [00:00<00:00, 125.25it/s]
2026-06-26 20:23:06,596 - INFO - HTTP Request: POST https://ea318a9e-a25e-4577-b965-253ad8328b1c.eu-central-1-0.aws.cloud.qdrant.io:6333/collections/nodejs_docs/points/query "HTTP/1.1 200 OK"
Batches: 100%|██████████| 1/1 [00:00<00:00, 57.62it/s]

Query: How do I set up an Express server with npm?

[Rank 1] rerank=7.934 | page 38 | chunk 153
Version 1.0 39 
Express 101 
To get started, add Express to your project. 
npm i express@4.16.4 
Next, you can require express. You get access to a single funct
--------------------------------------------------
[Rank 2] rerank=5.988 | page 108 | chunk 358
Version 1.0 109 
Lesson 4: Getting Started with Socket.io 
In this lesson, you’ll install and configure Socket.io. Socket.io comes with everything 
needed to se
--------------------------------------------------
[Rank 3] rerank=3.953 | page 42 | chunk 163
Setting up Handlebars 
Start by installing Handlebars in your project. 
npm i hbs@4.0.1 
From there, you’ll need to use app.set to set a value for the 'view eng
--------------------------------------------------
[Rank 4] rerank=3.921 | page 102 | chunk 342
Supertest was created by the Express team to allow you to easily test your Express apps. 
First up, install the module. 
npm i superte